# 13 — Skill Gap Engine
Set subtraction: required skills (via role mapping) minus employee current skills, weighted by importance (Data Value from IM scale).

In [1]:

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PROC = r'../data/processed'

ea = pd.read_csv(f'{PROC}/employee_attrition_processed.csv')
mapping_df = pd.read_csv(f'{PROC}/role_mapping.csv')
ess_im = pd.read_csv(f'{PROC}/essential_skills_processed.csv')
emp_skills = pd.read_csv(f'{PROC}/employee_skills_placeholder.csv')

print(f"Employees: {len(ea)}")
print(f"Role mappings: {len(mapping_df)}")
print(f"Essential skills (IM): {len(ess_im)}")
print(f"Employee current skills: {len(emp_skills)}")


Employees: 500
Role mappings: 24
Essential skills (IM): 9100
Employee current skills: 1979


In [2]:

# ── Build role → required skills lookup ──
# Key: SOC Code → {skill_name: importance_score}
role_to_soc = dict(zip(mapping_df['JobRole'], mapping_df['SOC_Code']))

# Required skills per SOC code: {element_name: data_value (importance, 1-5)}
soc_required_skills = {}
for soc, group in ess_im.groupby('O*NET-SOC Code'):
    # Deduplicate skills: if same element appears multiple times, take max importance
    skill_importance = group.groupby('Element Name')['Data Value'].max().to_dict()
    soc_required_skills[soc] = skill_importance

print(f"SOC codes with required skills: {len(soc_required_skills)}")
print(f"Sample required skills for first SOC: {dict(list(soc_required_skills.items())[0][1])}")


SOC codes with required skills: 910
Sample required skills for first SOC: {'Active Learning': 3.75, 'Active Listening': 4.0, 'Critical Thinking': 4.38, 'Learning Strategies': 3.12, 'Mathematics': 3.25, 'Monitoring': 4.0, 'Reading Comprehension': 4.12, 'Science': 1.62, 'Speaking': 4.25, 'Writing': 4.12}


In [3]:

# ── Build per-employee current skills lookup ──
emp_current_skills = emp_skills.groupby('employee_id')['current_skill'].apply(set).to_dict()
print(f"Employees with current skills: {len(emp_current_skills)}")


Employees with current skills: 500


In [4]:

# ── Compute skill gap per employee ──
# Gap = required skills NOT in current skills, weighted by importance
gap_records = []

for _, emp_row in ea.iterrows():
    emp_id = emp_row['EmployeeID']
    role = emp_row['JobRole']
    soc = role_to_soc.get(role)
    
    if soc is None or soc not in soc_required_skills:
        continue
    
    required = soc_required_skills[soc]  # {skill: importance}
    current = emp_current_skills.get(emp_id, set())
    
    for skill, importance in required.items():
        has_skill = skill in current
        gap_records.append({
            'employee_id': emp_id,
            'role': role,
            'soc_code': soc,
            'skill': skill,
            'importance': importance,
            'has_skill': has_skill,
            'is_gap': not has_skill
        })

skill_gap_df = pd.DataFrame(gap_records)
print(f"Skill gap analysis records: {len(skill_gap_df)}")
print(f"Total gaps: {skill_gap_df['is_gap'].sum()} ({skill_gap_df['is_gap'].mean():.1%} of all skill-employee pairs)")


Skill gap analysis records: 5000
Total gaps: 3021 (60.4% of all skill-employee pairs)


In [5]:

# ── Per-employee gap summary ──
emp_gap_summary = skill_gap_df.groupby('employee_id').agg(
    role=('role', 'first'),
    total_required=('skill', 'count'),
    gaps_count=('is_gap', 'sum'),
    weighted_gap_score=('importance', lambda x: x[skill_gap_df.loc[x.index, 'is_gap']].sum()),
    coverage_pct=('has_skill', 'mean')
).reset_index()

emp_gap_summary['gap_pct'] = emp_gap_summary['gaps_count'] / emp_gap_summary['total_required']
emp_gap_summary['weighted_gap_score'] = emp_gap_summary['weighted_gap_score'].round(2)
emp_gap_summary['coverage_pct'] = emp_gap_summary['coverage_pct'].round(3)

print(f"Employee gap summary shape: {emp_gap_summary.shape}")
print(emp_gap_summary.describe().round(2))
print(f"\nSample (top 5 worst gaps):")
print(emp_gap_summary.nlargest(5, 'weighted_gap_score')[['employee_id','role','gaps_count','weighted_gap_score','coverage_pct']])


Employee gap summary shape: (500, 7)
       employee_id  total_required  gaps_count  weighted_gap_score  \
count       500.00           500.0      500.00              500.00   
mean        250.50            10.0        6.04               21.07   
std         144.48             0.0        0.81                3.15   
min           1.00            10.0        5.00               14.13   
25%         125.75            10.0        5.00               18.37   
50%         250.50            10.0        6.00               21.18   
75%         375.25            10.0        7.00               23.63   
max         500.00            10.0        7.00               28.00   

       coverage_pct  gap_pct  
count        500.00   500.00  
mean           0.40     0.60  
std            0.08     0.08  
min            0.30     0.50  
25%            0.30     0.50  
50%            0.40     0.60  
75%            0.50     0.70  
max            0.50     0.70  

Sample (top 5 worst gaps):
     employee_id         

In [6]:

# ── Save ──
skill_gap_df.to_csv(f'{PROC}/skill_gap_detailed.csv', index=False)
emp_gap_summary.to_csv(f'{PROC}/employee_skill_gap_summary.csv', index=False)
print("Saved: skill_gap_detailed.csv, employee_skill_gap_summary.csv")


Saved: skill_gap_detailed.csv, employee_skill_gap_summary.csv


**Skill gap engine complete.** Set subtraction applied per employee per role. Gap scored by IM importance weights from O*NET.